# Debug: CFTC Commitments of Traders (`fetch_cftc.py`)

Step-by-step debug notebook for the CFTC COT flat-file pipeline.

**Pipeline:** `pipeline/fetch_cftc.py`  
**Output:** `data/cftc_cot.parquet`  
**Source:** CFTC public FTP — no API key required

---
Data is downloaded as annual ZIP archives from the CFTC website and filtered to  
the Henry Hub Natural Gas futures market (market code `023651`).

## 1. Imports & Dependencies

In [ ]:
import importlib
import io
import sys
import zipfile
from datetime import date
from pathlib import Path

required = ["pandas", "requests", "pyarrow"]
missing = []
for pkg in required:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

if missing:
    print(f"FAIL  Missing packages: {missing}")
else:
    import pandas as pd
    import requests
    print(f"PASS  All packages available — pandas {pd.__version__}")

## 2. Configuration

In [ ]:
ROLLING_YEARS = 5
NG_MARKET_CODE = "023651"  # Henry Hub Natural Gas futures (NYMEX)
CFTC_URL_TEMPLATE = "https://www.cftc.gov/files/dea/history/fut_disagg_txt_{year}.zip"
OUTPUT_PATH = Path("../data/cftc_cot.parquet")

current_year = date.today().year
years = list(range(current_year - ROLLING_YEARS, current_year + 1))

KEEP_COLUMNS = [
    "Market_and_Exchange_Names",
    "As_of_Date_In_Form_YYMMDD",
    "CFTC_Market_Code",
    "Open_Interest_All",
    "Prod_Merc_Positions_Long_All",
    "Prod_Merc_Positions_Short_All",
    "Swap__Positions_Long_All",
    "Swap__Positions_Short_All",
    "M_Money_Positions_Long_All",
    "M_Money_Positions_Short_All",
    "Other_Rept_Positions_Long_All",
    "Other_Rept_Positions_Short_All",
    "NonRept_Positions_Long_All",
    "NonRept_Positions_Short_All",
    "M_Money_Positions_Spread_All",
    "Swap__Positions_Spread_All",
]

print(f"NG market code : {NG_MARKET_CODE}")
print(f"Years to fetch : {years}")
print(f"Output path    : {OUTPUT_PATH.resolve()}")
print(f"Exists         : {OUTPUT_PATH.exists()}")

## 3. Connectivity Test — Download One Year's Archive

Download the most recent year's file and inspect the ZIP contents.

In [ ]:
# Use the previous year to be sure the file exists
test_year = current_year - 1
test_url = CFTC_URL_TEMPLATE.format(year=test_year)

print(f"Testing download: {test_url}")

try:
    resp = requests.get(test_url, timeout=60)
    print(f"Status        : {resp.status_code}")
    print(f"Content-Type  : {resp.headers.get('Content-Type', 'unknown')}")
    print(f"Content-Length: {resp.headers.get('Content-Length', 'unknown')} bytes")

    if resp.status_code == 404:
        print(f"FAIL  404 — file for {test_year} not found at CFTC")
        print(f"      URL: {test_url}")
    elif resp.status_code != 200:
        print(f"FAIL  Unexpected status {resp.status_code}")
    else:
        # Inspect ZIP contents
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            names = zf.namelist()
            print(f"\nPASS  ZIP downloaded successfully")
            print(f"      Files in archive: {names}")
            txt_files = [n for n in names if n.endswith(".txt")]
            print(f"      .txt files: {txt_files}")
except requests.Timeout:
    print("FAIL  Timed out — CFTC server may be slow, retry")
except Exception as e:
    print(f"FAIL  Exception: {e}")

## 4. Inspect Raw CSV Structure

In [ ]:
if resp.status_code != 200:
    print("SKIP — fix download first")
else:
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        txt_files = [n for n in zf.namelist() if n.endswith(".txt")]
        if not txt_files:
            print("FAIL  No .txt file found in ZIP")
        else:
            with zf.open(txt_files[0]) as f:
                raw = pd.read_csv(f, low_memory=False)

    print(f"Raw CSV shape : {raw.shape}")
    print(f"Total columns : {len(raw.columns)}")
    print()
    print("All column names:")
    for i, col in enumerate(raw.columns):
        print(f"  {i:3d}. {col}")

## 5. Locate Market Code Column & Filter to NG

In [ ]:
if resp.status_code != 200:
    print("SKIP")
else:
    # Find the market code column (name varies slightly across years)
    code_col = next(
        (c for c in raw.columns if "CFTC_Market_Code" in c or "Commodity_Code" in c),
        None,
    )

    if code_col is None:
        print("FAIL  Could not find market code column")
        print("      Columns containing 'Code':")
        print([c for c in raw.columns if "Code" in c or "code" in c])
    else:
        print(f"Market code column: '{code_col}'")
        print()

        # Show sample unique codes
        raw[code_col] = raw[code_col].astype(str).str.strip()
        print(f"Unique market codes (first 15): {sorted(raw[code_col].unique())[:15]}")
        print()

        ng_rows = raw[raw[code_col] == NG_MARKET_CODE]
        print(f"NG rows (code={NG_MARKET_CODE}): {len(ng_rows)}")

        if ng_rows.empty:
            print("FAIL  No NG rows found — check market code")
            print(f"      Searching for similar codes:")
            similar = raw[raw[code_col].str.contains("023", na=False)][code_col].unique()
            print(f"      {similar}")
        else:
            print(f"PASS  Found {len(ng_rows)} NG rows")
            print()
            print("Sample row:")
            print(ng_rows.iloc[0][code_col[:20] if len(code_col) > 20 else code_col])
            
            # Show the market name
            if "Market_and_Exchange_Names" in ng_rows.columns:
                names = ng_rows["Market_and_Exchange_Names"].unique()
                print(f"Market names: {names}")

## 6. Check Required Columns are Present

In [ ]:
if resp.status_code != 200 or ng_rows.empty:
    print("SKIP")
else:
    present = [c for c in KEEP_COLUMNS if c in ng_rows.columns]
    absent  = [c for c in KEEP_COLUMNS if c not in ng_rows.columns]

    print(f"Expected columns : {len(KEEP_COLUMNS)}")
    print(f"Present          : {len(present)}")
    print(f"Missing          : {len(absent)}")

    if absent:
        print(f"\nWARN  Missing columns: {absent}")
        print("      These will be silently skipped by the pipeline")
    else:
        print("\nPASS  All expected columns present")

## 7. Download All Years (Full Pipeline)

In [ ]:
def download_year(year: int) -> pd.DataFrame:
    url = CFTC_URL_TEMPLATE.format(year=year)
    try:
        r = requests.get(url, timeout=60)
    except requests.RequestException as exc:
        print(f"  WARNING: Network error for {year}: {exc}")
        return pd.DataFrame()

    if r.status_code == 404:
        print(f"  {year}: 404 (file not yet published)")
        return pd.DataFrame()
    if r.status_code != 200:
        print(f"  WARNING: CFTC returned {r.status_code} for {year}")
        return pd.DataFrame()

    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        txt_files = [n for n in zf.namelist() if n.endswith(".txt")]
        if not txt_files:
            print(f"  WARNING: No .txt in ZIP for {year}")
            return pd.DataFrame()
        with zf.open(txt_files[0]) as f:
            df = pd.read_csv(f, low_memory=False)

    code_col = next(
        (c for c in df.columns if "CFTC_Market_Code" in c or "Commodity_Code" in c),
        None,
    )
    if code_col is None:
        print(f"  WARNING: No market code column in {year}")
        return pd.DataFrame()

    df[code_col] = df[code_col].astype(str).str.strip()
    ng = df[df[code_col] == NG_MARKET_CODE].copy()
    if ng.empty:
        print(f"  {year}: no NG rows")
        return pd.DataFrame()

    cols_present = [c for c in KEEP_COLUMNS if c in ng.columns]
    return ng[cols_present].copy()


print(f"Downloading CFTC COT for years: {years}")
print()

year_frames = {}
for yr in years:
    print(f"  {yr} ...", end=" ", flush=True)
    df_yr = download_year(yr)
    if not df_yr.empty:
        year_frames[yr] = df_yr
        print(f"{len(df_yr)} NG rows")
    # (404 / empty cases already printed inside download_year)

print()
print(f"Years with data: {sorted(year_frames.keys())}")

if not year_frames:
    print("FAIL  No data downloaded — check network or CFTC URL template")
else:
    print(f"PASS  {len(year_frames)}/{len(years)} years downloaded")

## 8. Clean & Combine

In [ ]:
if not year_frames:
    print("SKIP")
else:
    combined_raw = pd.concat(year_frames.values(), ignore_index=True)
    print(f"Combined raw rows: {len(combined_raw)}")
    print()

    # Parse date
    date_col = "As_of_Date_In_Form_YYMMDD"
    if date_col in combined_raw.columns:
        combined_raw["date"] = pd.to_datetime(
            combined_raw[date_col].astype(str), format="%y%m%d", errors="coerce"
        )
        bad_dates = combined_raw["date"].isna().sum()
        if bad_dates > 0:
            print(f"WARN  {bad_dates} rows with unparseable dates")
            print("      Sample bad values:",
                  combined_raw.loc[combined_raw["date"].isna(), date_col].head(5).tolist())
        else:
            print(f"PASS  All dates parsed successfully")
        combined_raw = combined_raw.drop(columns=[date_col])
    else:
        print(f"FAIL  Date column '{date_col}' not found")

    # Coerce numeric columns
    skip_cols = {"date", "Market_and_Exchange_Names", "CFTC_Market_Code"}
    numeric_cols = [c for c in combined_raw.columns if c not in skip_cols]
    for col in numeric_cols:
        combined_raw[col] = pd.to_numeric(combined_raw[col], errors="coerce")

    # Derived net positions
    if {"M_Money_Positions_Long_All", "M_Money_Positions_Short_All"}.issubset(combined_raw.columns):
        combined_raw["managed_money_net"] = (
            combined_raw["M_Money_Positions_Long_All"] - combined_raw["M_Money_Positions_Short_All"]
        )
    if {"Prod_Merc_Positions_Long_All", "Prod_Merc_Positions_Short_All"}.issubset(combined_raw.columns):
        combined_raw["producer_net"] = (
            combined_raw["Prod_Merc_Positions_Long_All"] - combined_raw["Prod_Merc_Positions_Short_All"]
        )

    combined = combined_raw.sort_values("date").reset_index(drop=True)

    # Trim to rolling window
    cutoff = pd.Timestamp(date.today().replace(year=date.today().year - ROLLING_YEARS))
    combined = combined[combined["date"] >= cutoff].reset_index(drop=True)

    print(f"\nAfter trim: {len(combined)} rows (cutoff: {cutoff.date()})")
    print(f"Date range: {combined['date'].min().date()} → {combined['date'].max().date()}")
    print()
    print("Schema:")
    print(combined.dtypes.to_string())

## 9. Data Quality Checks

In [ ]:
if not year_frames:
    print("SKIP")
else:
    issues = []

    # Check for duplicate dates (COT is weekly, each date should appear once)
    dupes = combined["date"].duplicated().sum()
    if dupes > 0:
        issues.append(f"{dupes} duplicate report dates")

    # Check weekly cadence — should be ~7 days between reports
    gaps = combined["date"].diff().dt.days.dropna()
    irregular = gaps[gaps > 14]
    if not irregular.empty:
        issues.append(f"{len(irregular)} gaps > 14 days between weekly reports")

    # Check open interest is positive
    if "Open_Interest_All" in combined.columns:
        bad_oi = (combined["Open_Interest_All"] <= 0).sum()
        if bad_oi > 0:
            issues.append(f"{bad_oi} rows with non-positive open interest")

    # Check managed money net is within plausible range
    if "managed_money_net" in combined.columns:
        mm_max = combined["managed_money_net"].abs().max()
        if mm_max > 500_000:
            issues.append(f"managed_money_net max = {mm_max:,.0f} (unusually large)")

    if issues:
        print("WARN  Issues:")
        for i in issues:
            print(f"  - {i}")
    else:
        print("PASS  No data quality issues detected")

    print()
    print("Key statistics:")
    key_cols = [c for c in ["Open_Interest_All", "managed_money_net", "producer_net"] if c in combined.columns]
    print(combined[key_cols].describe().round(0).to_string())

## 10. Managed Money Positioning (Latest)

In [ ]:
if not year_frames or "managed_money_net" not in combined.columns:
    print("SKIP")
else:
    latest = combined.tail(8)[["date", "Open_Interest_All", "managed_money_net", "producer_net"]].copy()
    latest["mm_net_pct_oi"] = (
        latest["managed_money_net"] / latest["Open_Interest_All"] * 100
    ).round(1)

    print("Latest 8 weekly reports:")
    print(latest.to_string(index=False))
    print()

    latest_mm = combined["managed_money_net"].iloc[-1]
    percentile = (combined["managed_money_net"] < latest_mm).mean() * 100
    print(f"Current managed money net: {latest_mm:,.0f} contracts")
    print(f"Percentile vs 5yr history : {percentile:.0f}th")

## 11. Compare to Saved Parquet

In [ ]:
if OUTPUT_PATH.exists():
    saved = pd.read_parquet(OUTPUT_PATH)
    print(f"Saved parquet: {len(saved)} rows")
    print(f"Columns      : {list(saved.columns)}")
    print(f"Date range   : {saved['date'].min().date()} → {saved['date'].max().date()}")
    print()
    print("Latest 3 saved rows:")
    print(saved.tail(3)[["date", "Open_Interest_All", "managed_money_net", "producer_net"]].to_string())

    if year_frames:
        fresh_max = combined["date"].max()
        saved_max = saved["date"].max()
        lag = (fresh_max - saved_max).days
        if lag > 14:
            print(f"\nWARN  Saved parquet is {lag} days behind fresh fetch")
        else:
            print(f"\nPASS  Saved parquet is up to date (lag = {lag} days)")
else:
    print(f"INFO  No saved parquet at {OUTPUT_PATH} — run pipeline to generate it")

## 12. Write Output (optional)

In [ ]:
# if year_frames:
#     OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#     combined.to_parquet(OUTPUT_PATH, index=False)
#     print(f"Wrote {len(combined)} rows → {OUTPUT_PATH}")